# Indraneel - LSTM training and tuning on Colab

This notebook prepares UCI HAR, runs the baseline LSTM trainer, performs the Prompt 2 tuning sweep, and downloads the selected model and result files. Reusable model and tuning code stays in `src/train_lstm.py` and `src/tune_lstm.py`.

Before cloning, commit and push the current notebook and source changes to `indraneel-app`. For example: `git add src/train_lstm.py src/tune_lstm.py lstm_training_colab.ipynb INDRANEEL_PLAN.md`, then commit and push.

## 1. Select a GPU runtime

In Colab, choose **Runtime > Change runtime type > T4 GPU** (or another available GPU). The project uses TensorFlow 2.16.1 from `requirements.txt`; if Colab asks for a runtime restart after installation, restart and rerun from the clone step.

In [1]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/aryan-2206/Human_Activity_Recognition.git'
BRANCH = 'indraneel-app'
REPO_DIR = '/content/Human_Activity_Recognition'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

Working directory: /content/Human_Activity_Recognition


## 2. Install dependencies and verify the accelerator

Keep TensorFlow and scikit-learn aligned with the project environment so saved models load consistently.

In [2]:
%pip install -r requirements.txt
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

ERROR: Could not find a version that satisfies the requirement tensorflow==2.16.1 (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0, 2.22.0rc0)
ERROR: No matching distribution found for tensorflow==2.16.1
TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Run note: TensorFlow version mismatch

In this Colab run, `%pip install -r requirements.txt` could not install the pinned TensorFlow 2.16.1 under Python 3.13. The subsequent cells used the runtime-provided TensorFlow 2.21.0 (the saved model reports Keras 3.13.2) and training completed. Before merging this model into the shared demo, align the team environment: either use a Python runtime that supports TensorFlow 2.16.1 and rerun, or have the shared requirements owner update the TensorFlow pin and validate all models against it. Do not treat the failed install cell as a successful 2.16.1 install.

## 3. Prepare and inspect the shared data

`prepare_data()` downloads UCI HAR if needed, applies the train-fitted channel scaler, and writes the shared windows, labels, subjects, and label names under `data/processed/`. The generated files are ignored by Git, so they are created in this Colab clone.

In [3]:
from src.data import prepare_data

X_train, X_test, y_train, y_test, subjects_train, subjects_test = prepare_data()
print('Train:', X_train.shape, y_train.shape, subjects_train.shape)
print('Test: ', X_test.shape, y_test.shape, subjects_test.shape)

train: X=(7352, 128, 6), y=(7352,), counts={'WALKING': 1226, 'WALKING_UPSTAIRS': 1073, 'WALKING_DOWNSTAIRS': 986, 'SITTING': 1286, 'STANDING': 1374, 'LAYING': 1407}
test: X=(2947, 128, 6), y=(2947,), counts={'WALKING': 496, 'WALKING_UPSTAIRS': 471, 'WALKING_DOWNSTAIRS': 420, 'SITTING': 491, 'STANDING': 532, 'LAYING': 537}
Train: (7352, 128, 6) (7352,) (7352,)
Test:  (2947, 128, 6) (2947,) (2947,)


In [4]:
import json
import numpy as np
from pathlib import Path

label_names = json.loads(Path('data/processed/label_names.json').read_text())
assert X_train.ndim == 3 and X_train.shape[1:] == (128, 6)
assert X_test.ndim == 3 and X_test.shape[1:] == (128, 6)
assert len(X_train) == len(y_train) == len(subjects_train)
assert len(X_test) == len(y_test) == len(subjects_test)
print('Labels:', label_names)
print('Training subjects:', np.unique(subjects_train).size)

Labels: ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']
Training subjects: 21


## 4. Train and evaluate the LSTM

The script holds out about 15% of training subjects for validation, trains the two-layer LSTM with gradient clipping, early stopping, and learning-rate reduction, then evaluates on the official test split. It saves `models/lstm.keras`, `results/lstm_metrics.json`, and `results/lstm_curves.png`. The metric JSON includes the shared helper output plus `macro_f1` and `per_class_report` aliases required by the original common prompt.

In [5]:
!python src/train_lstm.py --epochs 40 --batch-size 128 --validation-fraction 0.15 --seed 42

[train_lstm] Preparing/loading UCI HAR data...
train: X=(7352, 128, 6), y=(7352,), counts={'WALKING': 1226, 'WALKING_UPSTAIRS': 1073, 'WALKING_DOWNSTAIRS': 986, 'SITTING': 1286, 'STANDING': 1374, 'LAYING': 1407}
test: X=(2947, 128, 6), y=(2947,), counts={'WALKING': 496, 'WALKING_UPSTAIRS': 471, 'WALKING_DOWNSTAIRS': 420, 'SITTING': 491, 'STANDING': 532, 'LAYING': 537}
W0000 00:00:1794425255.981031    4183 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
Model: "LSTM_HAR"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sensor_window (InputLayer)      │ (None, 128, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                  

## 5. Download artifacts and return them to the branch

Download these outputs and copy them into the matching paths in your local `indraneel-app` checkout. Keep files over about 50 MB on Drive and add a README link. Remove this notebook manually if the team chooses not to retain it in GitHub.

In [6]:
from pathlib import Path
from google.colab import files

output_paths = [
    Path('models/lstm.keras'),
    Path('results/lstm_metrics.json'),
    Path('results/lstm_curves.png'),
]
missing = [str(path) for path in output_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing expected training outputs: {missing}')
for path in output_paths:
    files.download(str(path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Prompt 2: compare LSTM variants

Run this after the team resolves the TensorFlow version mismatch noted above. The script compares the baseline, smaller hidden size, two dropout settings, and a bidirectional variant on the same subject-wise validation split. It selects by validation macro-F1, uses the official test set only for the selected configuration, and stops fitting when the 30-minute budget expires. It refreshes `models/lstm.keras`, `results/lstm_metrics.json`, and `results/lstm_curves.png`; the comparison table is `results/lstm_tuning.csv`.

In [ ]:
!python src/tune_lstm.py --epochs 20 --batch-size 128 --validation-fraction 0.15 --seed 42 --early-stopping-patience 4 --time-budget-minutes 30

## 7. Download the selected model and tuning record

After the tuning cell completes, download the refreshed model, metrics, curves, and CSV table. Copy these files into the same paths in your local `indraneel-app` checkout.

In [ ]:
from pathlib import Path
from google.colab import files

tuned_outputs = [
    Path('models/lstm.keras'),
    Path('results/lstm_metrics.json'),
    Path('results/lstm_curves.png'),
    Path('results/lstm_tuning.csv'),
]
missing = [str(path) for path in tuned_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Missing tuning outputs: {missing}')
for path in tuned_outputs:
    files.download(str(path))